# PII Redaction on Snowflake

This hands-on lab explores three approaches to PII (Personally Identifiable Information) redaction using Snowflake Cortex AI functions.

### What You'll Learn

1. **AI_REDACT** — Snowflake's built-in managed PII redaction function
2. **AI_COMPLETE Extract+Replace** — A custom approach using structured LLM output
3. **Pre-Computed Cache** — A hybrid pattern for sub-second query-time redaction

### Prerequisites

- Run `setup.sql` before starting this notebook (creates database, tables, UDF, sample data)
- Database: `PII_REDACTION_DEMO`
- Schema: `REDACTION`
- Warehouse: `PII_REDACTION_WH`

---
## Section 1: Connect & Explore Data

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql("USE DATABASE PII_REDACTION_DEMO").collect()
session.sql("USE SCHEMA REDACTION").collect()
session.sql("USE WAREHOUSE PII_REDACTION_WH").collect()
print("Connected to PII_REDACTION_DEMO.REDACTION")

In [ ]:
# Document type distribution
session.sql("""
    SELECT DOC_TYPE, COUNT(DISTINCT DOC_ID) AS NUM_DOCS, COUNT(*) AS NUM_CHUNKS
    FROM DOCUMENT_CHUNKS
    GROUP BY DOC_TYPE
    ORDER BY NUM_DOCS DESC
""").show()

In [ ]:
# Sample chunks with embedded PII
session.sql("""
    SELECT DOC_ID, CHUNK_INDEX, DOC_TYPE, LEFT(CHUNK_TEXT, 120) AS PREVIEW
    FROM DOCUMENT_CHUNKS
    WHERE CHUNK_INDEX <= 2
    ORDER BY DOC_ID
    LIMIT 10
""").show(max_width=130)

Notice the sample data contains names, emails, phone numbers, SSNs, addresses, and other PII embedded in realistic document chunks.

---
## Section 2: AI_REDACT — The Managed Approach

`AI_REDACT` is Snowflake's built-in function for detecting and redacting PII. It requires no setup — just call it on any text column.

### 2.1 Basic Redact Mode

The `'redact'` mode replaces detected PII with category labels like `[NAME]`, `[EMAIL]`, etc.

In [ ]:
session.sql("""
    SELECT
        DOC_ID,
        CHUNK_INDEX,
        AI_REDACT(CHUNK_TEXT, 'redact') AS REDACTED_TEXT
    FROM DOCUMENT_CHUNKS
    LIMIT 5
""").show(max_width=150)

### 2.2 Detect Mode

The `'detect'` mode returns a JSON array showing the location and type of each PII entity — useful for inspection without modification.

In [ ]:
session.sql("""
    SELECT
        DOC_ID,
        CHUNK_INDEX,
        AI_REDACT(CHUNK_TEXT, 'detect') AS DETECTED_PII
    FROM DOCUMENT_CHUNKS
    WHERE CHUNK_INDEX = 2
    LIMIT 5
""").show(max_width=200)

### 2.3 Category Filtering

You can restrict redaction to specific PII categories.

In [ ]:
session.sql("""
    SELECT
        DOC_ID,
        CHUNK_INDEX,
        AI_REDACT(CHUNK_TEXT, 'redact', {'categories': ['NAME', 'EMAIL']}) AS REDACTED_NAMES_EMAILS
    FROM DOCUMENT_CHUNKS
    WHERE CHUNK_INDEX = 2
    LIMIT 5
""").show(max_width=150)

### 2.4 Timing AI_REDACT on a Batch

In [ ]:
import time

start = time.time()
result = session.sql("""
    SELECT
        DOC_ID,
        CHUNK_INDEX,
        AI_REDACT(CHUNK_TEXT, 'redact') AS REDACTED_TEXT
    FROM DOCUMENT_CHUNKS
    LIMIT 50
""").collect()
elapsed_ai_redact = time.time() - start

print(f"AI_REDACT on 50 rows: {elapsed_ai_redact:.1f} seconds")
print(f"Rows processed: {len(result)}")

**AI_REDACT Limitations:**
- Combined input+output token limit of 4096 tokens
- Optimized for US-centric PII types (12 fixed categories)
- Generic labels (`[NAME]`, `[EMAIL]`) — no custom label formatting
- Best performance with English text

---
## Section 3: AI_COMPLETE Extract+Replace — The Custom Approach

This approach uses `AI_COMPLETE` with structured JSON output to extract PII entities, then applies a Python UDF for typed replacement. It offers full control over categories, labels, and model choice.

### 3.1 Structured JSON Extraction

We prompt the model to return PII entities as structured JSON with `phrase` and `field_type` fields.

In [ ]:
session.sql("""
    SELECT
        DOC_ID,
        CHUNK_INDEX,
        AI_COMPLETE(
            'mistral-large2',
            CONCAT(
                'Identify any PII from the following content such as persons full names, ',
                'email addresses, phone numbers, passport numbers, driver licence numbers, ',
                'SSNs, ITINs, bank account numbers, credit card numbers, or any other ',
                'sensitive or classified identifiers. ',
                'If no PII is found, return an empty array for redacted_items. ',
                'If uncertain whether a value is PII, leave it unchanged. ',
                'Return an array of redacted items with phrase and field_type. ',
                CHUNK_TEXT
            ),
            {
                'response_format': {
                    'type': 'json',
                    'schema': {
                        'type': 'object',
                        'properties': {
                            'redacted_items': {
                                'type': 'array',
                                'items': {
                                    'type': 'object',
                                    'properties': {
                                        'phrase': {'type': 'string'},
                                        'field_type': {'type': 'string'}
                                    }
                                }
                            }
                        },
                        'required': ['redacted_items']
                    }
                },
                'temperature': 0,
                'max_tokens': 1000
            }
        ) AS PII_JSON
    FROM DOCUMENT_CHUNKS
    WHERE CHUNK_INDEX = 2
    LIMIT 3
""").show(max_width=200)

### 3.2 Apply the REDACT_PII UDF

The `REDACT_PII` UDF (created by `setup.sql`) replaces each detected phrase with a typed label like `[NAME REDACTED]`, `[EMAIL REDACTED]`, etc.

In [ ]:
session.sql("""
    WITH extracted AS (
        SELECT
            DOC_ID,
            CHUNK_INDEX,
            CHUNK_TEXT,
            AI_COMPLETE(
                'mistral-large2',
                CONCAT(
                    'Identify any PII from the following content such as persons full names, ',
                    'email addresses, phone numbers, passport numbers, driver licence numbers, ',
                    'SSNs, ITINs, bank account numbers, credit card numbers, or any other ',
                    'sensitive or classified identifiers. ',
                    'If no PII is found, return an empty array for redacted_items. ',
                    'If uncertain whether a value is PII, leave it unchanged. ',
                    'Return an array of redacted items with phrase and field_type. ',
                    CHUNK_TEXT
                ),
                {
                    'response_format': {
                        'type': 'json',
                        'schema': {
                            'type': 'object',
                            'properties': {
                                'redacted_items': {
                                    'type': 'array',
                                    'items': {
                                        'type': 'object',
                                        'properties': {
                                            'phrase': {'type': 'string'},
                                            'field_type': {'type': 'string'}
                                        }
                                    }
                                }
                            },
                            'required': ['redacted_items']
                        }
                    },
                    'temperature': 0,
                    'max_tokens': 1000
                }
            ) AS PII_JSON
        FROM DOCUMENT_CHUNKS
        LIMIT 5
    )
    SELECT
        DOC_ID,
        CHUNK_INDEX,
        REDACT_PII(CHUNK_TEXT, PARSE_JSON(PII_JSON)) AS REDACTED_TEXT
    FROM extracted
""").show(max_width=150)

### 3.3 Timing AI_COMPLETE Extract+Replace

Same 50 rows as AI_REDACT for a fair comparison.

In [ ]:
start = time.time()
result = session.sql("""
    WITH extracted AS (
        SELECT
            DOC_ID,
            CHUNK_INDEX,
            CHUNK_TEXT,
            AI_COMPLETE(
                'mistral-large2',
                CONCAT(
                    'Identify any PII from the following content such as persons full names, ',
                    'email addresses, phone numbers, passport numbers, driver licence numbers, ',
                    'SSNs, ITINs, bank account numbers, credit card numbers, or any other ',
                    'sensitive or classified identifiers. ',
                    'If no PII is found, return an empty array for redacted_items. ',
                    'If uncertain whether a value is PII, leave it unchanged. ',
                    'Return an array of redacted items with phrase and field_type. ',
                    CHUNK_TEXT
                ),
                {
                    'response_format': {
                        'type': 'json',
                        'schema': {
                            'type': 'object',
                            'properties': {
                                'redacted_items': {
                                    'type': 'array',
                                    'items': {
                                        'type': 'object',
                                        'properties': {
                                            'phrase': {'type': 'string'},
                                            'field_type': {'type': 'string'}
                                        }
                                    }
                                }
                            },
                            'required': ['redacted_items']
                        }
                    },
                    'temperature': 0,
                    'max_tokens': 1000
                }
            ) AS PII_JSON
        FROM DOCUMENT_CHUNKS
        LIMIT 50
    )
    SELECT
        DOC_ID,
        CHUNK_INDEX,
        REDACT_PII(CHUNK_TEXT, PARSE_JSON(PII_JSON)) AS REDACTED_TEXT
    FROM extracted
""").collect()
elapsed_ai_complete = time.time() - start

print(f"AI_COMPLETE Extract+Replace on 50 rows: {elapsed_ai_complete:.1f} seconds")
print(f"Rows processed: {len(result)}")

**AI_COMPLETE Extract+Replace Advantages:**
- Any model (mistral-large2, claude, llama, etc.)
- Custom PII categories (domain-specific: MRN, policy numbers, internal IDs)
- Custom labels (`[NAME REDACTED]`, `[SSN REDACTED]` vs generic `[NAME]`)
- No hard token limit (model-dependent, typically 32K+)
- Multi-language support via model capability

---
## Section 4: Head-to-Head Comparison

Let's run both approaches on 100 rows and compare results.

In [ ]:
# AI_REDACT on 100 rows
start = time.time()
redact_results = session.sql("""
    SELECT
        DOC_ID,
        CHUNK_INDEX,
        AI_REDACT(CHUNK_TEXT, 'redact') AS REDACTED_TEXT
    FROM DOCUMENT_CHUNKS
    LIMIT 100
""").collect()
time_ai_redact_100 = time.time() - start
print(f"AI_REDACT (100 rows): {time_ai_redact_100:.1f} seconds")

In [ ]:
# AI_COMPLETE Extract+Replace on 100 rows
start = time.time()
complete_results = session.sql("""
    WITH extracted AS (
        SELECT
            DOC_ID,
            CHUNK_INDEX,
            CHUNK_TEXT,
            AI_COMPLETE(
                'mistral-large2',
                CONCAT(
                    'Identify any PII from the following content such as persons full names, ',
                    'email addresses, phone numbers, passport numbers, driver licence numbers, ',
                    'SSNs, ITINs, bank account numbers, credit card numbers, or any other ',
                    'sensitive or classified identifiers. ',
                    'If no PII is found, return an empty array for redacted_items. ',
                    'If uncertain whether a value is PII, leave it unchanged. ',
                    'Return an array of redacted items with phrase and field_type. ',
                    CHUNK_TEXT
                ),
                {
                    'response_format': {
                        'type': 'json',
                        'schema': {
                            'type': 'object',
                            'properties': {
                                'redacted_items': {
                                    'type': 'array',
                                    'items': {
                                        'type': 'object',
                                        'properties': {
                                            'phrase': {'type': 'string'},
                                            'field_type': {'type': 'string'}
                                        }
                                    }
                                }
                            },
                            'required': ['redacted_items']
                        }
                    },
                    'temperature': 0,
                    'max_tokens': 1000
                }
            ) AS PII_JSON
        FROM DOCUMENT_CHUNKS
        LIMIT 100
    )
    SELECT
        DOC_ID,
        CHUNK_INDEX,
        REDACT_PII(CHUNK_TEXT, PARSE_JSON(PII_JSON)) AS REDACTED_TEXT
    FROM extracted
""").collect()
time_ai_complete_100 = time.time() - start
print(f"AI_COMPLETE Extract+Replace (100 rows): {time_ai_complete_100:.1f} seconds")

### Side-by-Side Comparison

Compare the same chunk redacted by each approach.

In [ ]:
comparison = session.sql("""
    SELECT
        dc.DOC_ID,
        dc.CHUNK_INDEX,
        dc.CHUNK_TEXT AS ORIGINAL,
        AI_REDACT(dc.CHUNK_TEXT, 'redact') AS AI_REDACT_OUTPUT,
        REDACT_PII(
            dc.CHUNK_TEXT,
            PARSE_JSON(
                AI_COMPLETE(
                    'mistral-large2',
                    CONCAT(
                        'Identify any PII from the following content such as persons full names, ',
                        'email addresses, phone numbers, passport numbers, driver licence numbers, ',
                        'SSNs, ITINs, bank account numbers, credit card numbers, or any other ',
                        'sensitive or classified identifiers. ',
                        'If no PII is found, return an empty array for redacted_items. ',
                        'If uncertain whether a value is PII, leave it unchanged. ',
                        'Return an array of redacted items with phrase and field_type. ',
                        dc.CHUNK_TEXT
                    ),
                    {
                        'response_format': {
                            'type': 'json',
                            'schema': {
                                'type': 'object',
                                'properties': {
                                    'redacted_items': {
                                        'type': 'array',
                                        'items': {
                                            'type': 'object',
                                            'properties': {
                                                'phrase': {'type': 'string'},
                                                'field_type': {'type': 'string'}
                                            }
                                        }
                                    }
                                },
                                'required': ['redacted_items']
                            }
                        },
                        'temperature': 0,
                        'max_tokens': 1000
                    }
                )
            )
        ) AS AI_COMPLETE_OUTPUT
    FROM DOCUMENT_CHUNKS dc
    WHERE dc.CHUNK_INDEX = 2
    LIMIT 3
""").to_pandas()

for _, row in comparison.iterrows():
    print(f"\n{'='*80}")
    print(f"DOC: {row['DOC_ID']} | Chunk: {row['CHUNK_INDEX']}")
    print(f"{'='*80}")
    print(f"\nORIGINAL:\n{row['ORIGINAL'][:200]}")
    print(f"\nAI_REDACT:\n{row['AI_REDACT_OUTPUT'][:200]}")
    print(f"\nAI_COMPLETE:\n{row['AI_COMPLETE_OUTPUT'][:200]}")

### Comparison Summary

| Dimension | AI_REDACT | AI_COMPLETE Extract+Replace |
|-----------|-----------|---------------------------|
| Setup | Zero (built-in function) | Prompt + UDF |
| Speed | Fast (optimized) | Varies by model |
| Cost | Fixed per call | Depends on model + tokens |
| Categories | 12 fixed US PII types | Custom (any category) |
| Token limit | 4096 combined I/O | Model-dependent |
| Labels | Generic ([NAME], [EMAIL]) | Custom ([FIRST_NAME REDACTED]) |
| Languages | Best with English | Multi-language via model |

In [ ]:
print(f"""
Performance Summary (this run):
  AI_REDACT (100 rows):              {time_ai_redact_100:.1f}s
  AI_COMPLETE Extract+Replace (100):  {time_ai_complete_100:.1f}s
""")

---
## Section 5: Pre-Computed Cache Pattern

For production workloads where query-time latency matters, extract PII entities at ingestion time and cache them. At query time, just JOIN with the cache — no LLM call needed.

### 5.1 Populate the PII Entity Cache

The `PII_ENTITY_CACHE` table was created by `setup.sql`. Let's populate it using AI_COMPLETE.

In [ ]:
start = time.time()
session.sql("""
    INSERT INTO PII_ENTITY_CACHE (DOC_ID, CHUNK_INDEX, PII_ENTITIES, CHUNK_HASH)
    SELECT
        DOC_ID,
        CHUNK_INDEX,
        PARSE_JSON(
            AI_COMPLETE(
                'mistral-large2',
                CONCAT(
                    'Identify any PII from the following content such as persons full names, ',
                    'email addresses, phone numbers, passport numbers, driver licence numbers, ',
                    'SSNs, ITINs, bank account numbers, credit card numbers, or any other ',
                    'sensitive or classified identifiers. ',
                    'If no PII is found, return an empty array for redacted_items. ',
                    'If uncertain whether a value is PII, leave it unchanged. ',
                    'Return an array of redacted items with phrase and field_type. ',
                    CHUNK_TEXT
                ),
                {
                    'response_format': {
                        'type': 'json',
                        'schema': {
                            'type': 'object',
                            'properties': {
                                'redacted_items': {
                                    'type': 'array',
                                    'items': {
                                        'type': 'object',
                                        'properties': {
                                            'phrase': {'type': 'string'},
                                            'field_type': {'type': 'string'}
                                        }
                                    }
                                }
                            },
                            'required': ['redacted_items']
                        }
                    },
                    'temperature': 0,
                    'max_tokens': 1000
                }
            )
        ) AS PII_ENTITIES,
        MD5(CHUNK_TEXT) AS CHUNK_HASH
    FROM DOCUMENT_CHUNKS
""").collect()
elapsed_cache_populate = time.time() - start

print(f"Cache populated in {elapsed_cache_populate:.1f} seconds (one-time ingestion cost)")

### 5.2 Inspect Cached Entities

In [ ]:
session.sql("""
    SELECT
        DOC_ID,
        CHUNK_INDEX,
        PII_ENTITIES,
        EXTRACTED_AT
    FROM PII_ENTITY_CACHE
    WHERE PII_ENTITIES:redacted_items[0] IS NOT NULL
    LIMIT 5
""").show(max_width=200)

### 5.3 Query-Time Redaction via Cache JOIN

At query time, we JOIN with the cache and apply the UDF — zero LLM calls.

In [ ]:
start = time.time()
cached_results = session.sql("""
    SELECT
        dc.DOC_ID,
        dc.CHUNK_INDEX,
        REDACT_PII(dc.CHUNK_TEXT, cache.PII_ENTITIES) AS REDACTED_TEXT
    FROM DOCUMENT_CHUNKS dc
    INNER JOIN PII_ENTITY_CACHE cache
        ON dc.DOC_ID = cache.DOC_ID
        AND dc.CHUNK_INDEX = cache.CHUNK_INDEX
        AND MD5(dc.CHUNK_TEXT) = cache.CHUNK_HASH
    LIMIT 100
""").collect()
time_cached = time.time() - start

print(f"Cached redaction (100 rows): {time_cached:.1f} seconds")
print(f"Rows processed: {len(cached_results)}")

In [ ]:
# Show sample cached output
session.sql("""
    SELECT
        dc.DOC_ID,
        dc.CHUNK_INDEX,
        LEFT(dc.CHUNK_TEXT, 100) AS ORIGINAL_PREVIEW,
        LEFT(REDACT_PII(dc.CHUNK_TEXT, cache.PII_ENTITIES), 100) AS REDACTED_PREVIEW
    FROM DOCUMENT_CHUNKS dc
    INNER JOIN PII_ENTITY_CACHE cache
        ON dc.DOC_ID = cache.DOC_ID
        AND dc.CHUNK_INDEX = cache.CHUNK_INDEX
    WHERE cache.PII_ENTITIES:redacted_items[0] IS NOT NULL
    LIMIT 5
""").show(max_width=120)

### 5.4 Three-Way Performance Comparison

In [ ]:
print(f"""
Three-Way Performance Comparison (100 rows):
{'='*50}
  AI_REDACT (live):              {time_ai_redact_100:>6.1f}s
  AI_COMPLETE Extract+Replace:   {time_ai_complete_100:>6.1f}s
  Pre-Computed Cache (JOIN+UDF): {time_cached:>6.1f}s
{'='*50}

The cached approach eliminates LLM latency entirely at query time.
Ingestion cost was {elapsed_cache_populate:.1f}s (amortized, runs once per document).
""")

---
## Section 6: Decision Framework

### When to Use Each Approach

**Use AI_REDACT when:**
- Standard US PII categories (names, emails, phones, SSNs, etc.) are sufficient
- You want the simplest possible implementation (zero setup)
- Text fits within 4096 combined input+output tokens
- English-language content

**Use AI_COMPLETE Extract+Replace when:**
- Custom PII categories needed (MRNs, policy numbers, internal IDs)
- Non-English text requires redaction
- You want typed/custom labels (`[SSN REDACTED]` vs generic `[PII]`)
- Text exceeds AI_REDACT's 4096 token limit
- You need to control model choice for cost or quality tradeoffs

**Use Pre-Computed Cache when:**
- Query-time latency is critical (sub-second requirement)
- Documents are relatively stable (not changing every request)
- You can afford compute at ingestion time
- High query volume on the same documents (amortizes extraction cost)

### Hybrid Architecture

In production, combine approaches:
1. **AI_REDACT** for quick ad-hoc redaction of small text
2. **Pre-Computed Cache** for your core document corpus (extracted at ingestion)
3. **AI_COMPLETE live fallback** for new/changed documents not yet in cache

---
## Summary

| Step | What We Did |
|------|-------------|
| **Explore** | Examined 200 synthetic document chunks with embedded PII across 20 documents |
| **AI_REDACT** | Used Snowflake's built-in function in redact, detect, and filtered modes |
| **AI_COMPLETE** | Built a custom extract+replace pipeline with structured JSON output |
| **Compare** | Ran head-to-head on 100 rows comparing speed, output quality, and flexibility |
| **Cache** | Populated a PII entity cache for sub-second query-time redaction |
| **Decide** | Established a decision framework for choosing the right approach |

### Key Takeaways

1. **AI_REDACT is the fastest path** — Zero setup, good for standard US PII categories
2. **AI_COMPLETE gives full control** — Custom categories, models, labels, and languages
3. **Pre-computed cache eliminates query-time latency** — Best for production workloads with stable documents
4. **They're complementary, not competing** — Use the right tool for each part of your pipeline

In [ ]:
# Optional: Clean up (uncomment to run)
# session.sql("DROP DATABASE IF EXISTS PII_REDACTION_DEMO CASCADE").collect()
# print("Lab resources cleaned up.")